# Newport News, VA — Land Value Tax Model

**City real-estate levy, revenue-neutral 4:1 split-rate, all current exemptions preserved.**

Newport News is one of Virginia's **independent cities** — it is not part of any county and is its own
county-equivalent for tax and Census purposes (FIPS 51700). Virginia assesses all real property at
**100% of fair market value** (Va. Code § 58.1-3201); there is no assessment ratio or equalization
factor. The city levies a single real-estate tax rate on that full assessed value, so the current tax is
simply `assessed value × rate`. This model holds the FY2026 rate of **$1.18 per $100** (11.8 mills) and
restructures the city real-estate levy into a revenue-neutral 4:1 split-rate (land assessed value taxed
4× improvement assessed value). Virginia's uniformity clause (Const. Art. X § 1) currently bars a true
split-rate without enabling authority, so this is an illustrative revenue-neutral restructuring of the
existing base.

| | |
|---|---|
| Geographic scope | City of Newport News (independent city), FIPS 51700 |
| Levy modeled | City real-estate tax only, single citywide rate on 100% FMV |
| Reform | Revenue-neutral 4:1 split-rate (land taxed 4× improvements) |
| Assessment | 100% of fair market value (Va. Code § 58.1-3201); no ratio |
| Rate | $1.18 per $100 = 11.8 mills (FY2026 Adopted Budget, held flat) |
| Validation | Modeled taxable assessed value vs. FY2026 Adopted Budget tax-year-2026 figure of **$23,539,239,130** (p. 61) |
| Data source | City of Newport News `Operational/Parcel` MapServer (assessor parcels, current land/improvement values, use & class codes, owner, government flag) |

**Two jurisdiction-specific data treatments (documented in Sections 3 & 5b):**
- **Public service corporations excluded.** Dominion (Virginia Electric & Power), railroads (CSX,
  Chesapeake & Ohio), Verizon, Virginia Natural Gas, Colonial Pipeline, etc. are **assessed by the
  Virginia State Corporation Commission, not the local assessor**, and the budget explicitly excludes
  public-service-corporation taxes from the current real-estate levy. In this parcel file they also carry
  obvious placeholder values (Dominion's $277,427,942 is stamped on all 25 of its parcels → $6.94B of
  phantom value). They are dropped entirely.
- **Exemptions reconstructed from the government flag + use codes.** The GIS file has no single
  tax-exempt flag, so fully-exempt parcels are identified as `GOVERNMENT = 'Y'` (federal, state — incl.
  Christopher Newport University, city, schools) **or** a curated set of constitutionally/charitably
  exempt land uses (churches, cemeteries, hospitals, museums, charitable social-service agencies,
  colleges, schools, libraries, fire/police stations).

## Section 1 — Imports and Setup

In [ ]:
import sys
import json
import os
import io
import time
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'newport_news'
STATE_FIPS = '51'                 # Virginia
COUNTY_FIPS = '700'               # Newport News (independent city = county-equivalent 51700)
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

CITY_MILLAGE = 11.8               # $1.18 per $100 of assessed value (FY2026, held flat)
# FY2026 Adopted Budget, p.61 — tax-year-2026 total taxable real-estate assessed value ("levy")
OFFICIAL_TAXABLE_AV = 23_539_239_130

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print(f"Rate: ${CITY_MILLAGE/10:.2f} per $100 = {CITY_MILLAGE} mills (100% FMV)")
print(f"Split ratio: {LAND_IMPROVEMENT_RATIO:.0f}:1 (land:improvement)")

## Section 2 — Load Parcel Data

Parcels come from the City of Newport News **`Operational/Parcel` MapServer** (the assessor parcel base).
The layer carries the current land/improvement values, use and class codes, owner, the government flag,
and geometry — everything the model needs. Geometry is requested in EPSG:4326 (`outSR=4326`) so the
Census centroid join works directly. The full file (~54k parcels) is paginated 1,000 at a time and cached.

| Concept | Column | Notes |
|---|---|---|
| Parcel ID | `PARCELID` | |
| Current land value | `CNTLNDVAL` | Fair market value (100% FMV) |
| Current improvement value | `CNTIMPVAL` | Buildings/structures at 100% FMV |
| Use description | `USEDSCRP` | Granular assessor land use |
| Class code / desc | `CLASSCD` / `CLASSDSCRP` | R=single-family, O=condo, M=multi 2-4, A=apartment 5+, C=commercial, I=industrial, T=trailer court |
| Owner | `OWNERNME1` | Used to identify public-service corporations |
| Government flag | `GOVERNMENT` | `'Y'` = government-owned (exempt) |
| Living units | `LIVUNIT` | |

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
PARCEL_LAYER = ('https://maps.nnva.gov/gis/rest/services/'
                'Operational/Parcel/MapServer/0/query')
FIELDS = ('PARCELID,OWNERNME1,USECD,USEDSCRP,CLASSCD,CLASSDSCRP,CNTLNDVAL,CNTIMPVAL,'
          'PRVLNDVAL,PRVIMPVAL,GOVERNMENT,VACANT,CITYOWNED,LIVUNIT,RESYRBLT,RESFLRAREA,'
          'SITEADDRESS,NGHBRHDCD,ZONE,CENSUSTRACT,CENSUSBLOCK,STATEDAREA')

def fetch_newport_news_parcels():
    """Paginate the Newport News Parcel MapServer as GeoJSON in EPSG:4326."""
    total = requests.get(PARCEL_LAYER, params={
        'f': 'json', 'where': '1=1', 'returnCountOnly': 'true'}, timeout=60).json()['count']
    parts, offset = [], 0
    while offset < total:
        r = requests.get(PARCEL_LAYER, params={
            'f': 'geojson', 'where': '1=1', 'outFields': FIELDS, 'returnGeometry': 'true',
            'outSR': 4326, 'resultOffset': offset, 'resultRecordCount': 1000}, timeout=180)
        r.raise_for_status()
        g = gpd.read_file(io.StringIO(r.text))
        if len(g) == 0:
            break
        parts.append(g)
        offset += len(g)
        time.sleep(0.1)
    gdf = pd.concat(parts, ignore_index=True)
    return gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    gdf = fetch_newport_news_parcels()
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels")

for c in ['CNTLNDVAL', 'CNTIMPVAL']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0).clip(lower=0)
gdf['taxable_land_value'] = gdf['CNTLNDVAL']
gdf['taxable_improvement_value'] = gdf['CNTIMPVAL']
gdf['total_value'] = gdf['CNTLNDVAL'] + gdf['CNTIMPVAL']

print(f"Total assessed value (all parcels): ${gdf['total_value'].sum():,.0f}")
print(f"Median total value: ${gdf['total_value'].median():,.0f}")
print(f"Parcels with $0 improvement (vacant candidates): {(gdf['CNTIMPVAL'] <= 0).sum():,}")

## Section 3 — Classify Parcels

**Public service corporations (excluded).** Identified by owner name. These are State-Corporation-
Commission-assessed, excluded from the local real-estate levy per the FY2026 budget, and carry placeholder
values (Dominion's identical $277.4M on all 25 parcels). They are dropped before modeling.

**Fully-exempt parcels (held out).** No single exempt flag exists, so exempt = `GOVERNMENT = 'Y'`
(federal incl. Fort Eustis, state incl. Christopher Newport University, city, public schools) **or** a
curated set of exempt land uses (places of worship, cemeteries, hospitals, museums, charitable social-
service agencies, colleges, K-12 schools, public buildings, libraries, fire/police stations, post office).
Per repo norm they are held out of the revenue-neutral solver and excluded from the export and charts.

**Property categories** are built from `USEDSCRP` with `CLASSCD` as the authority for the multi-family
unit-count split (M = 2-4 dwellings → Small Multi-Family; A = apartments 5+ → Large Multi-Family), plus
the standard override that any parcel with $0 improvement value becomes **Vacant Land**.

In [ ]:
own = gdf['OWNERNME1'].fillna('').str.upper()

# --- Public service corporations: SCC-assessed, excluded from the local levy ---
PSC_PATTERN = (
    r'VIRGINIA ELECTRIC|DOMINION ENERGY|DOMINION VIRGINIA|VEPCO|VIRGINIA NATURAL GAS|COLUMBIA GAS|'
    r'CSX|CHESAPEAKE & OHIO|CHESAPEAKE AND OHIO|NORFOLK SOUTHERN|NORFOLK & WESTERN|RAILWAY CO|'
    r'RAILROAD CO|VERIZON|BELL ATLANTIC|CHESAPEAKE & POTOMAC|COX COMMUNICATIONS|'
    r'VIRGINIA-AMERICAN WATER|COLONIAL PIPELINE|PLANTATION PIPE|PIPELINE CO'
)
gdf['is_psc'] = own.str.contains(PSC_PATTERN, regex=True)

# --- Fully-exempt: government-owned OR exempt land use ---
gov = gdf['GOVERNMENT'].fillna('').str.strip().eq('Y')
use_norm = gdf['USEDSCRP'].fillna('').str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()
EXEMPT_USES = {
    'CHURCH/SYNAGOGUE/OTHER PLACES OF WORSHIP', 'CHURCH/SYAGOGUE/OTHER PLACES OF WORSHIP',
    'CEMETERY/MAUSOLEUM', 'HOSPITAL', 'MUSEUM', 'NON-GOVERNMENTAL SOCIAL SERVICE AGENCIES',
    'UNIVERSITY/COLLEGE/BUSINESS SCHOOL', 'PUBLIC/PRIVATE SCHOOL K-12', 'PUBLIC BUILDINGS',
    'LIBRARY', 'FIRE STATION', 'POLICE STATION', 'POST OFFICE',
}
gdf['full_exmp'] = ((gov | use_norm.isin(EXEMPT_USES)) & ~gdf['is_psc']).astype(int)

print(f"Public service corporations (excluded): {int(gdf['is_psc'].sum()):,} "
      f"(${gdf.loc[gdf['is_psc'], 'total_value'].sum():,.0f})")
print(f"Fully-exempt parcels (held out):        {int(gdf['full_exmp'].sum()):,} "
      f"(${gdf.loc[gdf['full_exmp'] == 1, 'total_value'].sum():,.0f})")
print(f"  of which government-owned: ${gdf.loc[gov & ~gdf['is_psc'], 'total_value'].sum():,.0f}")
print(f"  of which exempt land use:  ${gdf.loc[use_norm.isin(EXEMPT_USES) & ~gov & ~gdf['is_psc'], 'total_value'].sum():,.0f}")


def categorize(use, cls):
    use = ('' if (use is None or (isinstance(use, float) and pd.isna(use))) else str(use)).upper().replace('  ', ' ').strip()
    cls = ('' if (cls is None or (isinstance(cls, float) and pd.isna(cls))) else str(cls)).upper().strip()
    OFFICE = {'PROFESSIONAL OFFICE', 'PROFFESIONAL OFFICE', 'SMALL CONTRACTOR PROFESSIONAL OFFICE',
              'OFFICE AND WAREHOUSE', 'BANK', 'MEDICAL CLINIC', 'DOCTOR/DENTIST/MEDICAL LABORATORY'}
    RETAIL = {'SPECIALTY SHOP', 'GENERAL RETAIL', 'NEIGHBORHOOD SHOPPING CENTER', 'COMMUNITY SHOPPING CENTER',
              'RESTAURANT', 'FAST FOOD RESTAURANT', 'CONVENIENCE STORE', 'CONVENIENCE STORE WITH GAS PUMPS',
              'GROCERY STORE', 'PHARMACY', 'HARDWARE STORE', 'ELECTRONICS STORE', 'AUTO PARTS',
              'AUTO SALES OR RENTAL', 'AUTO SALES', 'AUTO SALES OR RENTAL (COMMERCIAL)', 'AUTO SERVICE',
              'PERSONAL SERVICE', 'LAUNDRY/DRYCLEANER', 'CAR WASH', 'GAS PUMPS WITH CAR WASH',
              'FLEA MARKET/PAWN SHOP', 'HEALTH GYM/SPA', 'VETERINARIAN/KENNEL', 'FUNERAL HOME OR MORTUARY'}
    IND = {'ENCLOSED MANUFACTURING/PROCESSING', 'OPEN AIR INDUSTRIAL', 'ASSEMBLY OPERATIONS', 'MACHINE SHOP',
           'WHOLESALE DISTRIBUTION FACILITY', 'MINI-WAREHOUSE FACILITY', 'WAREHOUSING', 'MOVING/STORAGE',
           'CONTRACTOR SERVICE', 'RESEARCH/DEVELOPMENT FACILITY', 'PORT RELATED FACILITIES AND ACTIVITIES',
           'COAL TERMINAL', 'JUNKYARD/RECYCLING', 'EQUIPMENT SALES AND RENTAL', 'MARINE SALES AND SERVICE',
           'NEWSPAPER/MAGAZINE PUBLISHING/PRINT SHOP', 'TOWING & STORAGE', 'MOBILE HOME SALES AND SERVICE',
           'COMMUNICATION TOWER'}
    OTHER_COMM = {'THEATER', 'COMMERCIAL RECREATION', 'PRIVATE RECREATION CENTER', 'CITY RECREATION CENTER',
                  'ADULT USE/NIGHT CLUB', 'LODGE/CLUB', 'ADULT/CHILD CARE CENTER', 'SPECIALITY SCHOOL',
                  'PARK', 'TRANSPORTATION CENTER/ANNEX', 'TRANSPORTATION CENTER/ANNEX(INDUSTRIAL)'}
    if 'PARKING' in use:
        return 'Transportation - Parking'
    if use in ('VACANT', 'OPEN SPACE', 'WETLANDS', 'STREETS/ROW'):
        return 'Vacant Land'
    if use == '2 OR MORE USES':
        return 'Mixed Use'
    if use == 'HOTEL/MOTEL':
        return 'Hotel'
    if cls == 'A' or 'MULTI FAMILY' in use:
        return 'Large Multi-Family (5+ units)'
    if cls == 'M':
        return 'Small Multi-Family (2-4 units)'
    if cls == 'O' or 'CONDOMINIUM' in use or use == 'COMDOMINIUM':
        return 'Condominium'
    if cls == 'T' or 'MOBILE HOME' in use:
        return 'Other Residential'
    if use in ('BOARDING HOUSE/DORMITORY/GROUP HOME', 'CONVALESCENT/NURSING HOME', 'BED AND BREAKFAST/TOURIST HOME'):
        return 'Other Residential'
    if cls == 'R':
        return 'Townhome / Rowhouse' if ('ATTACH' in use and 'DETA' not in use) else 'Single Family Residential'
    if 'SINGLE FAMILY ATTACHED' in use or use == 'PRD (ATTACHED)':
        return 'Townhome / Rowhouse'
    if 'SINGLE FAMILY DETA' in use or use in ('PRD (DETACHED)', 'SINGEL FAMILY DETATCHED'):
        return 'Single Family Residential'
    if use in OFFICE:
        return 'Office / Commercial Condo'
    if use in RETAIL:
        return 'Retail / General Commercial'
    if use in IND or cls == 'I':
        return 'Industrial'
    if use in OTHER_COMM:
        return 'Other Commercial'
    if cls == 'C':
        return 'Commercial'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = [categorize(u, c) for u, c in zip(gdf['USEDSCRP'], gdf['CLASSCD'])]
# Override: $0 improvement value -> Vacant Land, regardless of use code
gdf.loc[gdf['CNTIMPVAL'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print()
print(gdf.loc[~gdf['is_psc'], 'PROPERTY_CATEGORY'].value_counts().to_string())

## Section 4 — Current Tax + Validation (Gate 1)

Virginia assesses at 100% of fair market value, so the current real-estate tax is simply
`assessed value × rate`, with no assessment ratio:

```
current_tax = (CNTLNDVAL + CNTIMPVAL) × 11.8 / 1000     (taxable parcels only)
```

Public-service-corporation parcels are dropped first. The control is the **FY2026 Adopted Budget
tax-year-2026 taxable assessed value of $23,539,239,130** (p. 61). A small positive gap is expected: a
handful of arguably-exempt parcels (e.g. the Mariners' Museum park, the airport commission, some
continuing-care communities) are conservatively kept taxable, and the budget figure is net of partial
relief/deferral programs (elderly, disabled, ~$8.5M disabled-veteran) that are not flagged per-parcel in
the GIS data.

In [ ]:
gdf['millage_rate'] = CITY_MILLAGE

# Drop public service corporations entirely (out of local levy scope)
n_psc = int(gdf['is_psc'].sum())
gdf = gdf[~gdf['is_psc']].copy()

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

modeled_taxable_av = gdf.loc[gdf['full_exmp'] == 0, 'total_value'].sum()
gap_pct = (modeled_taxable_av / OFFICIAL_TAXABLE_AV - 1) * 100
print(f"Dropped {n_psc:,} public service corporation parcels.")
print(f"Modeled taxable assessed value: ${modeled_taxable_av:,.0f}")
print(f"Official (FY2026 budget, TY2026): ${OFFICIAL_TAXABLE_AV:,.0f}")
print(f"Gap: {gap_pct:+.2f}%")
print(f"Modeled gross city real-estate levy: ${current_revenue:,.0f}  "
      f"(budget gross ~${OFFICIAL_TAXABLE_AV*CITY_MILLAGE/1000/1e6:,.1f}M; net collections ~$267.7M after relief)")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"

## Section 5 — Split-Rate Model

Revenue-neutral 4:1 split-rate on the current land and improvement assessed values. Fully-exempt parcels
(and any with no assessed value) are held out of the solver and excluded from the export and charts.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
no_value = (gdf['full_exmp'] == 0) & (gdf['total_value'] <= 0)
n_novalue = int(no_value.sum())

gdf = gdf[(gdf['full_exmp'] == 0) & (gdf['total_value'] > 0)].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt + {n_novalue:,} zero-value parcels (excluded from model, export, charts).")
print(f"Modeled parcels: {len(gdf):,}")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}")
print(f"  (current flat rate {CITY_MILLAGE} mills; land taxed {LAND_IMPROVEMENT_RATIO:.0f}x improvements)")
print(f"Revenue neutrality: ${new_revenue:,.0f} vs target ${gdf['current_tax'].sum():,.0f} "
      f"(gap {(new_revenue/gdf['current_tax'].sum()-1)*100:+.4f}%)")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Newport News — 4:1 Split-Rate Tax Impact (city real-estate levy)')

## Section 5b — Read the Results (Gate 5 Artifact Scan)

Check building shares by category against priors, and watch for the known artifacts (Seattle commercial
placeholder values, Maricopa flat residential land ratio, St. Paul condo token land).

In [ ]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
summary = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_low_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(summary.round(3).to_string())

# Flat-ratio check (Maricopa canary): does single-family land share genuinely vary per parcel?
sfr = d[(d['PROPERTY_CATEGORY'] == 'Single Family Residential') & (d['taxable_improvement_value'] > 0)].copy()
sfr['land_share'] = sfr['taxable_land_value'] / (sfr['taxable_land_value'] + sfr['taxable_improvement_value'])
print(f"\nSFR land-share spread (flat-ratio check): "
      f"p10={sfr['land_share'].quantile(.1):.3f}  median={sfr['land_share'].median():.3f}  "
      f"p90={sfr['land_share'].quantile(.9):.3f}  stdev={sfr['land_share'].std():.3f}  "
      f"distinct land values={sfr['taxable_land_value'].nunique():,}")

## Section 6 — Exploration Chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values().plot.barh(ax=ax)
ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'Newport News — Median Tax Change % by Category ({LAND_IMPROVEMENT_RATIO:.0f}:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('saved category_preview.png')

## Section 7 — Census Join + Export

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

## Validation Summary

| Check | Result |
|---|---|
| Revenue match (Gate 1) | Modeled taxable assessed value vs. FY2026 Adopted Budget TY2026 figure ($23,539,239,130) — see Section 4 output (gap < 2%). Rate $1.18/$100 confirmed from the budget. |
| Distribution (Gate 2) | Vacant Land & parking large increases; apartments / office / condo decrease; detached single-family roughly neutral — see Section 5 |
| Parcel count | ~52.7k modeled; held out fully-exempt + dropped public-service-corporation parcels (see Sections 3-5) |
| Census coverage (Gate 3) | See Section 7 output |
| PNGs generated (Gate 4) | See `analysis/reports/newport_news/` |
| Artifact scan (Gate 5) | Dominion/PSC placeholder ($277.4M ×25 = $6.94B phantom) excluded; single-family land share genuinely varies per parcel (no Maricopa flat-ratio artifact); no Seattle commercial-placeholder or St. Paul condo-token-land artifact |

### Modeling choices & limitations

- **City real-estate levy only.** Newport News is an independent city, so there is no overlapping county
  levy; this models the single citywide real-estate tax on 100% of fair market value at $1.18/$100.
- **Public service corporations excluded.** Dominion (Virginia Electric & Power), railroads, Verizon,
  Virginia Natural Gas, Colonial Pipeline, etc. are assessed by the Virginia State Corporation Commission
  and excluded from the local real-estate levy; in this file they also carry placeholder values
  (Dominion's $277,427,942 stamped on all 25 parcels). They are dropped entirely.
- **Exemptions reconstructed.** The GIS file has no single tax-exempt flag, so fully-exempt parcels are
  identified as `GOVERNMENT = 'Y'` plus a curated set of exempt land uses (worship, cemetery, hospital,
  museum, charitable, college, school, public building, library, fire/police, post office). This
  reconstructs the assessor's exempt roll closely (modeled taxable base within ~1.5% of the budget figure)
  but is not parcel-for-parcel identical.
- **Partial relief not flagged.** Virginia elderly/disabled and disabled-veteran relief (~$9.1M combined,
  per the budget) reduces collections but is not flagged per parcel, so the model operates on full assessed
  values. This is the main reason the modeled gross levy slightly exceeds budgeted net collections.
- **No assessment ratio.** Virginia assesses at 100% FMV (Va. Code § 58.1-3201); current tax is a direct
  `value × rate`, so percentage results are exact for the modeled base.
- **Legality.** Virginia's uniformity clause currently bars a true split-rate without state enabling
  authority; this is an illustrative revenue-neutral restructuring of the existing tax base.